# LangGraph Basics — Hands-On Notebook

Source: `Section2/Section2_LangChain_Recap_and_LangGraph.md`

Contains Programs 1–6 from the section, each in its own cell.
All programs that call an LLM (Programs 1, 3, 4) now use a real OpenAI model via `ChatOpenAI` — configure `.env` first.
Programs 5–6 are pure deterministic Python/graph logic and need no API key.

## Setup — load environment variables

Run this once at the top so any real-model cells below can pick up credentials from `.env`.

In [11]:
from dotenv import load_dotenv
load_dotenv()

False

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = ""

## Program 1 — A summarisation chain (real LLM call)

LCEL chain: `prompt | model | parser`. Uses a real OpenAI model via `ChatOpenAI`.

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# --- the model ---
model = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.3)

prompt = ChatPromptTemplate.from_template(
    "You are a credit analyst. Summarise this loan note in ONE line:\n\n{note}"
)

chain = prompt | model | StrOutputParser()   # <-- this is LangChain Expression Language

result = chain.invoke({"note": "Credit score 762, FOIR 33%, no defaults, salaried, 8 yrs experience"})
print(result)

Low-risk borrower with a strong credit score, stable income, and no history of defaults.


## Program 2 — Defining and inspecting tools (no API key)

In [14]:
from langchain_core.tools import tool

@tool
def get_credit_score(pan: str) -> int:
    """Fetch the CIBIL credit score for a customer using their PAN number.
    Use this whenever a credit decision needs a bureau score."""
    fake_bureau = {"ABCDE1234F": 762, "XYZAB9876K": 640}
    return fake_bureau.get(pan, 700)

@tool
def calculate_foir(monthly_income: float, existing_emi: float) -> float:
    """Calculate FOIR (Fixed Obligation to Income Ratio) as a percentage.
    Use this to check whether the applicant's existing EMIs are within bank policy."""
    return round((existing_emi / monthly_income) * 100, 1)

# What the LLM actually "sees":
for t in (get_credit_score, calculate_foir):
    print(f"name   : {t.name}")
    print(f"desc   : {t.description}")
    print(f"schema : {t.args}\n")

# Calling a tool directly (this is what the agent does for you):
print("score:", get_credit_score.invoke({"pan": "ABCDE1234F"}))
print("foir :", calculate_foir.invoke({"monthly_income": 90000, "existing_emi": 30000}))

name   : get_credit_score
desc   : Fetch the CIBIL credit score for a customer using their PAN number.
Use this whenever a credit decision needs a bureau score.
schema : {'pan': {'title': 'Pan', 'type': 'string'}}

name   : calculate_foir
desc   : Calculate FOIR (Fixed Obligation to Income Ratio) as a percentage.
Use this to check whether the applicant's existing EMIs are within bank policy.
schema : {'monthly_income': {'title': 'Monthly Income', 'type': 'number'}, 'existing_emi': {'title': 'Existing Emi', 'type': 'number'}}

score: 762
foir : 33.3


## Program 3 — Memory via checkpointer (real LLM call)

In [15]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.3)

def call_model(state: MessagesState) -> dict:
    # state["messages"] already holds the FULL history for this thread
    return {"messages": [chat.invoke(state["messages"])]}

builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_edge(START, "model")
builder.add_edge("model", END)

app = builder.compile(checkpointer=InMemorySaver())   # <-- memory switch

config = {"configurable": {"thread_id": "customer-101"}}   # <-- the memory key

app.invoke({"messages": [{"role": "user", "content": "My name is Mani"}]}, config)
result = app.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config)

for m in result["messages"]:
    print(f"{m.type:6} | {m.content}")

human  | My name is Mani
ai     | Nice to meet you, Mani! How can I assist you today?
human  | What is my name?
ai     | Your name is Mani.


## Program 4 — The three concepts combined into one agent (NEEDS AN API KEY)

Reuses `get_credit_score` and `calculate_foir` from Program 2 — run that cell first.

In [16]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
# reuse get_credit_score and calculate_foir from Program 2

agent = create_agent(
    model="openai:gpt-3.5-turbo",          # or "bedrock_converse:<model-id>"
    tools=[get_credit_score, calculate_foir],
    system_prompt=(
        "You are a loan eligibility assistant for an Indian retail bank. "
        "Policy: minimum credit score 700, maximum FOIR 50%. "
        "Always fetch the score and compute FOIR before deciding."
    ),
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "app-9001"}}
out = agent.invoke(
    {"messages": [{"role": "user", "content":
        "PAN ABCDE1234F, monthly income 90000, existing EMI 30000. Eligible?"}]},
    config,
)
print(out["messages"][-1].text)

The customer has a credit score of 762, which meets the minimum requirement of 700. Additionally, the Fixed Obligation to Income Ratio (FOIR) is 33.3%, which is within the bank's policy limit of 50%. Therefore, the customer is eligible for a loan.


## Program 5 — The first DAG (no API key, fully tested)

Loan pre-screening: fetch bureau score → compute FOIR → apply bank policy → branch to approve or reject.

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

# Directed Acyclic Graph - DAG
# ---------- 1. STATE : the shared dict every node reads and writes ----------
class LoanState(TypedDict):
    applicant: str
    monthly_income: float
    emi_outstanding: float
    credit_score: int
    foir: float
    decision: str
    reason: str


# ---------- 2. NODES : plain functions, state in -> partial dict out ----------
def fetch_credit_score(state: LoanState) -> dict:
    """Stand-in for a real CIBIL API call."""
    fake_bureau = {"Ravi Kumar": 762, "Anitha S": 640}
    score = fake_bureau.get(state["applicant"], 700)
    print(f"[fetch_credit_score] score = {score}")
    return {"credit_score": score}          # merged into state


def compute_foir(state: LoanState) -> dict:
    foir = round((state["emi_outstanding"] / state["monthly_income"]) * 100, 1)
    print(f"[compute_foir] FOIR = {foir}%")
    return {"foir": foir}


def decide(state: LoanState) -> dict:
    """Bank policy lives here — deterministic, not LLM-decided."""
    if state["credit_score"] < 700:
        return {"decision": "REJECT",
                "reason": f"Credit score {state['credit_score']} below cut-off 700"}
    if state["foir"] > 50:
        return {"decision": "REJECT",
                "reason": f"FOIR {state['foir']}% exceeds policy cap 50%"}
    return {"decision": "APPROVE",
            "reason": f"Score {state['credit_score']}, FOIR {state['foir']}% within policy"}


def approve_note(state: LoanState) -> dict:
    print(f"[approve_note] {state['applicant']} APPROVED — {state['reason']}")
    return {}


def reject_note(state: LoanState) -> dict:
    print(f"[reject_note] {state['applicant']} REJECTED — {state['reason']}")
    return {}


# ---------- 3. ROUTER : decides which edge to take ----------
def route(state: LoanState) -> Literal["approve_note", "reject_note"]:
    return "approve_note" if state["decision"] == "APPROVE" else "reject_note"


# ---------- 4. BUILD THE GRAPH ----------
builder = StateGraph(LoanState)

builder.add_node("fetch_credit_score", fetch_credit_score)
builder.add_node("compute_foir", compute_foir)
builder.add_node("decide", decide)
builder.add_node("approve_note", approve_note)
builder.add_node("reject_note", reject_note)

builder.add_edge(START, "fetch_credit_score")
builder.add_edge("fetch_credit_score", "compute_foir")
builder.add_edge("compute_foir", "decide")
builder.add_conditional_edges(
    "decide", route,
    {"approve_note": "approve_note", "reject_note": "reject_note"},
)
builder.add_edge("approve_note", END)
builder.add_edge("reject_note", END)

graph = builder.compile(checkpointer=InMemorySaver())


# ---------- 5. RUN IT ----------
applicants = [
    ("Ravi Kumar", 90000, 30000),   # good score, good FOIR  -> APPROVE
    ("Anitha S",   60000, 20000),   # low score              -> REJECT
    ("Meena R",    50000, 30000),   # ok score, FOIR too high-> REJECT
]

for name, income, emi in applicants:
    print("-" * 55)
    out = graph.invoke(
        {"applicant": name, "monthly_income": income, "emi_outstanding": emi},
        config={"configurable": {"thread_id": name}},
    )
    print(f"FINAL: {out['decision']} | {out['reason']}")

# ---------- 6. BONUS: the graph draws itself ----------
print("\n--- Mermaid diagram of this graph ---")
print(graph.get_graph().draw_mermaid())

## Program 6 — Human-in-the-loop (no API key, fully tested)

Pause the graph, wait for an officer, resume exactly where it stopped.

In [19]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command


class State(TypedDict):
    applicant: str
    amount: int
    status: str


def prepare(state: State) -> dict:
    print(f"[prepare] disbursal of Rs.{state['amount']} for {state['applicant']}")
    return {"status": "PENDING_APPROVAL"}


def officer_approval(state: State) -> dict:
    # execution STOPS here and control returns to the caller
    decision = interrupt({"question": "Approve disbursal?", "amount": state["amount"]})
    return {"status": "APPROVED" if decision == "yes" else "REJECTED"}


def disburse(state: State) -> dict:
    print(f"[disburse] final status = {state['status']}")
    return {}


builder = StateGraph(State)
builder.add_node("prepare", prepare)
builder.add_node("officer_approval", officer_approval)
builder.add_node("disburse", disburse)
builder.add_edge(START, "prepare")
builder.add_edge("prepare", "officer_approval")
builder.add_edge("officer_approval", "disburse")
builder.add_edge("disburse", END)

graph = builder.compile(checkpointer=InMemorySaver())   # HITL REQUIRES a checkpointer

config = {"configurable": {"thread_id": "loan-77"}}

# --- first call: runs until the interrupt, then returns ---
paused = graph.invoke({"applicant": "Ravi Kumar", "amount": 500000}, config)
print("PAUSED AT:", paused["__interrupt__"])

# --- the officer decides (minutes or days later, different process, same thread_id) ---
final = graph.invoke(Command(resume=(input())), config)
print("FINAL:", final)

[prepare] disbursal of Rs.500000 for Ravi Kumar
PAUSED AT: [Interrupt(value={'question': 'Approve disbursal?', 'amount': 500000}, id='c2136701065faa6314ca17204331380f')]
[disburse] final status = REJECTED
FINAL: {'applicant': 'Ravi Kumar', 'amount': 500000, 'status': 'REJECTED'}
